In [1]:
import torch 
import triton 
import triton.language as tl
from triton.runtime import driver 

DEVICE = triton.runtime.driver.active.get_active_torch_device()


def is_hip():
    return triton.runtime.driver.active.get_current_target().backend == "hip"


def is_cdna():
    return is_hip() and triton.runtime.driver.active.get_current_target().arch in ('gfx940', 'gfx941', 'gfx942',
                                                                                   'gfx90a')


In [2]:
def naive_softmax(x):

    x_max = x.max(dim=1)[0]

    z = x- x_max[:,None]

    numerator = torch.exp(z)
    denominator = numerator.sum(dim=1)

    z = numerator/denominator[:,None]

    return z


In [3]:
@triton.jit
def softmax_kernel(output_ptr,
                   input_ptr,
                   output_row_stride,
                   input_row_stride,
                   n_row,
                   n_col,
                   BLOCK_SIZE: tl.constexpr,
                   num_stages: tl.constexpr
                ):
    row_start = tl.program_id(0)
    row_step = tl.num_programs(0)

    for row_idx in tl.range(row_start,n_row,step=row_step,num_stages=num_stages):
        row_start_ptr = input_ptr + row_idx*input_row_stride

        col_offest  = tl.arange(0,BLOCK_SIZE) 
        input_ptrs = row_start_ptr + col_offest
        mask = col_offest<n_col

        row = tl.load(input_ptrs,mask=mask,other=-float('inf'))
        
        row_max = tl.max(row,axis=0)

        row_max = row - row_max

        numerator = tl.exp(row_max)
        denominator = tl.sum(numerator,axis=0)

        output = numerator/denominator

        output_row_start_ptr = output_ptr+row_idx*output_row_stride
        output_ptrs = output_row_start_ptr+col_offest

        tl.store(output_ptrs,output,mask=mask)


In [4]:

properties = driver.active.utils.get_device_properties(DEVICE.index)
NUM_SM = properties["multiprocessor_count"]
NUM_REGS = properties["max_num_regs"]
SIZE_SMEM = properties["max_shared_mem"]
WARP_SIZE = properties["warpSize"]
target = triton.runtime.driver.active.get_current_target()
kernels = {}

def softmax(x):
    n_rows,n_cols = x.shape

    BLOCK_SIZE = triton.next_power_of_2(n_cols)

    num_warps = 8

    num_stages = 4 if SIZE_SMEM > 200000 else 2

    y = torch.empty_like(x)

    kernel = softmax_kernel.warmup(y, x, y.stride(0), x.stride(0), n_row= n_rows, n_col = n_cols, BLOCK_SIZE=BLOCK_SIZE,
                                   num_stages=num_stages, num_warps=num_warps, grid=(1, ))

    kernel._init_handles()
    n_regs = kernel.n_regs
    size_smem = kernel.metadata.shared

    if is_hip():
        NUM_GPRS = NUM_REGS
        if is_cdna():
            NUM_GPRS = NUM_REGS *2 
        MAX_NUM_THREADS = properties["max_threads_per_sm"]
        max_num_waves = MAX_NUM_THREADS// WARP_SIZE
        occupancy = min(NUM_GPRS// WARP_SIZE // n_regs, max_num_waves) // num_warps

    else:
        occupancy  = NUM_REGS // (n_regs*WARP_SIZE*num_warps)
    occupancy = min(occupancy,SIZE_SMEM//size_smem)
    num_programs = NUM_SM * occupancy

    num_programs = min(num_programs,n_rows)

    kernel[(num_programs, 1, 1)](y, x, y.stride(0), x.stride(0), n_rows, n_cols, BLOCK_SIZE, num_stages)
    return y

In [5]:
torch.manual_seed(0)
x = torch.randn(1823, 781, device=DEVICE)
y_triton = softmax(x)
y_torch = torch.softmax(x, axis=1)
assert torch.allclose(y_triton, y_torch), (y_triton, y_torch)